加载模型。
这里有几个细节：
- load_model_and_processor() 使用官方自定义模型类。
- 默认采用 BF16。
- 默认采用 flash_attention_2。
- 默认 model.eval()。
- 如果模型权重已经在 Hugging Face 缓存中，会优先使用缓存。

In [4]:
import sys
import torch
import importlib

from pathlib import Path
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path(
    "/home/20261024_1/Workspace/qyh/VLM-test"
)

# 1. Xiaomi-Lens-0 源码目录
XIAOMI_REPO = PROJECT_ROOT / "models" / "Xiaomi-Lens-0"
XIAOMI_SRC = XIAOMI_REPO / "src"

# 2. CV-Bench Adapter 目录
ADAPTER_DIR = (
    PROJECT_ROOT
    / "evaluation"
    / "cv-bench"
    / "xiaomi_lens"
)

# 3. 检查路径是否存在
assert XIAOMI_SRC.is_dir(), f"源码目录不存在: {XIAOMI_SRC}"
assert (XIAOMI_SRC / "xiaomi_lens" / "__init__.py").is_file()
assert ADAPTER_DIR.is_dir()

# 4. 添加两个目录到 Python 搜索路径
sys.path.insert(0, str(XIAOMI_SRC))
sys.path.insert(0, str(ADAPTER_DIR))

importlib.invalidate_caches()

# 5. 先验证 Xiaomi 模型包能否导入
import xiaomi_lens

from xiaomi_lens.constants import MASK_TOKEN
from xiaomi_lens.model.modeling_xiaomi_lens import (
    XiaomiLensQwen3_5ForConditionalGeneration
)

print("Xiaomi package:", xiaomi_lens.__file__)
print("MASK_TOKEN:", MASK_TOKEN)
print("Model class:", XiaomiLensQwen3_5ForConditionalGeneration)

# 6. 导入之前 CV-Bench 使用的 Adapter
from xiaomi_lens_adapter import XiaomiLensAdapter
adapter = XiaomiLensAdapter(
    model_path="SeerRay-Lab/Xiaomi-Lens-0",
    device="cuda:0",
    attn_implementation="flash_attention_2",
)
print("Adapter imported successfully!")

load_info = adapter.load()
print("模型加载成功！")
print(load_info)

Xiaomi package: /home/20261024_1/Workspace/qyh/VLM-test/models/Xiaomi-Lens-0/src/xiaomi_lens/__init__.py
MASK_TOKEN: <|mask|>
Model class: <class 'xiaomi_lens.model.modeling_xiaomi_lens.XiaomiLensQwen3_5ForConditionalGeneration'>
Adapter imported successfully!


Loading weights: 100%|██████████| 749/749 [00:03<00:00, 242.71it/s]


模型加载成功！
{'model_name': 'Xiaomi-Lens-0', 'model_path': '/home/20261024_1/.cache/huggingface/hub/models--SeerRay-Lab--Xiaomi-Lens-0/snapshots/68f57cdde1a19f751e78b18af04efa8ca8bd7fce', 'attn_implementation': 'flash_attention_2', 'dtype': 'bfloat16', 'load_seconds': 5.343169471248984, 'vram_after_load': {'allocated_gib': 8.609443187713623, 'reserved_gib': 8.62109375, 'peak_allocated_gib': 8.609453201293945, 'peak_reserved_gib': 8.62109375, 'gpu_used_gib': 23.66290283203125, 'gpu_total_gib': 94.9600830078125}, 'eos_token_ids': {248044, 248046}, 'think_close_token_id': 248069, 'mask_token_id': 248077}


In [5]:
def test_xiaomi(image, question, mode="off", max_new_tokens=2048):

    # 1. 构建输入
    prepared = adapter.build_inputs(
        image=image,
        instruction=question,
        mode=mode,
    )

    # 2. 验证 Thinking 模式是否真正生效
    print("Thinking mode:", mode)
    print("enable_thinking:", prepared["enable_thinking"])
    print("Prompt verified:", prepared["mode_verified"])

    if not prepared["mode_verified"]:
        raise RuntimeError("Thinking 模式验证失败！")

    # 3. 执行推理
    result = adapter.generate(
        prepared=prepared,
        mode=mode,
        max_new_tokens=max_new_tokens,
    )

    # 4. 分离 Thinking 与最终回答
    raw_text = result["raw_generation"]

    if mode == "on":
        _, separator, answer = raw_text.partition("</think>")

        if not separator:
            answer = "[未检测到 </think>，可能尚未完成思考]"
    else:
        answer = raw_text

    # 5. 输出结果
    print("\n" + "=" * 60)
    print("最终回答")
    print("=" * 60)
    print(answer.strip())

    print("\n" + "=" * 60)
    print("性能统计")
    print("=" * 60)

    print("生成 Token:", result["generated_tokens"])
    print("Thinking Token:", result["thinking_tokens"])
    print("Answer Token:", result["answer_tokens"])
    print("生成耗时:", f"{result['generate_seconds']:.3f}s")
    print("生成速度:", f"{result['tokens_per_second']:.2f} tokens/s")
    print("Peak VRAM:", f"{result['vram']['peak_allocated_gib']:.2f} GiB")
    print("Finish reason:", result["finish_reason"])
    return result

### 测试1

1. 尝试中文+非思考模式。模型能够正确回答。

In [9]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1. 这本书的书脊是什么主要颜色？
2. 如果书脊上有可辨认的文字，请准确抄写书名。
3. 简要描述这本书的位置。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，第 2 行、第 2 列的书是《Music》。

1. 这本书的书脊主要颜色是**白色**（或米白色）。
2. 书脊上的文字是：**Music**。
3. 这本书位于书架的**第二行、第二列**，在《Space》的右侧和《Animals》的左侧。<|im_end|>

性能统计
生成 Token: 82
Thinking Token: 0
Answer Token: 82
生成耗时: 2.267s
生成速度: 36.18 tokens/s
Peak VRAM: 17.79 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，第 2 行、第 2 列的书是《Music》。\n\n1. 这本书的书脊主要颜色是**白色**（或米白色）。\n2. 书脊上的文字是：**Music**。\n3. 这本书位于书架的**第二行、第二列**，在《Space》的右侧和《Animals》的左侧。<|im_end|>',
 'generated_tokens': 82,
 'thinking_tokens': 0,
 'answer_tokens': 82,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 2.2666295950766653,
 'tokens_per_second': 36.17706226818523,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.459603309631348,
  'reserved_gib': 17.85546875,
  'peak_allocated_gib': 17.790788650512695,
  'peak_reserved_gib': 17.85546875,
  'gpu_used_gib': 23.16552734375,
  'gpu_total_gib': 94.9600830078125}}

尝试中文+非思考模式+直接回答指令。模型能正确回答。

In [10]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
请直接输出答案，而不要输出你的分析过程。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
1. 这本书的书脊主要颜色是白色。
2. 书名是“Music”。
3. 这本书位于第 2 行、第 2 列。<|im_end|>

性能统计
生成 Token: 39
Thinking Token: 0
Answer Token: 39
生成耗时: 1.134s
生成速度: 34.38 tokens/s
Peak VRAM: 17.80 GiB
Finish reason: eos


{'raw_generation': '1. 这本书的书脊主要颜色是白色。\n2. 书名是“Music”。\n3. 这本书位于第 2 行、第 2 列。<|im_end|>',
 'generated_tokens': 39,
 'thinking_tokens': 0,
 'answer_tokens': 39,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 1.134426181903109,
 'tokens_per_second': 34.37861415942794,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.459604263305664,
  'reserved_gib': 17.85546875,
  'peak_allocated_gib': 17.797080516815186,
  'peak_reserved_gib': 17.85546875,
  'gpu_used_gib': 23.16552734375,
  'gpu_total_gib': 94.9600830078125}}

### 测试5

1. 中文+非思考模式。翻车了。

In [12]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，从左到右依次排列书籍：
1. 黄色书脊《苏东坡传》
2. 浅蓝色书脊《数学Ⅱ(女孩)》
3. 浅粉色书脊《数学Ⅱ(女孩)》
4. 浅紫色书脊《数学Ⅱ(女孩)》
5. 红色书脊《现代汉语词典》
6. 绿色书脊《图解电的基础知识入门》
7. 绿色书脊《图解电机基础知识入门》
8. 绿色书脊《图解机械基础知识入门》
9. 白色书脊《电磁场基本原理66课》
10. 白色书脊《电工电路基本原理66课》
11. 白色书脊《电子电路基本原理66课》
12. 白色书脊《苏东坡新传》（上）
13. 白色书脊《苏东坡新传》（下）
14. 蓝色书脊《新英汉小词典》

第14本书是《新英汉小词典》，书脊主要颜色为蓝色。

1. 这本书的书脊主要颜色是蓝色。
2. 书名：新英汉小词典
3. 位置：第1行，从左数第14本。<|im_end|>

性能统计
生成 Token: 280
Thinking Token: 0
Answer Token: 280
生成耗时: 8.164s
生成速度: 34.30 tokens/s
Peak VRAM: 19.02 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，从左到右依次排列书籍：\n1. 黄色书脊《苏东坡传》\n2. 浅蓝色书脊《数学Ⅱ(女孩)》\n3. 浅粉色书脊《数学Ⅱ(女孩)》\n4. 浅紫色书脊《数学Ⅱ(女孩)》\n5. 红色书脊《现代汉语词典》\n6. 绿色书脊《图解电的基础知识入门》\n7. 绿色书脊《图解电机基础知识入门》\n8. 绿色书脊《图解机械基础知识入门》\n9. 白色书脊《电磁场基本原理66课》\n10. 白色书脊《电工电路基本原理66课》\n11. 白色书脊《电子电路基本原理66课》\n12. 白色书脊《苏东坡新传》（上）\n13. 白色书脊《苏东坡新传》（下）\n14. 蓝色书脊《新英汉小词典》\n\n第14本书是《新英汉小词典》，书脊主要颜色为蓝色。\n\n1. 这本书的书脊主要颜色是蓝色。\n2. 书名：新英汉小词典\n3. 位置：第1行，从左数第14本。<|im_end|>',
 'generated_tokens': 280,
 'thinking_tokens': 0,
 'answer_tokens': 280,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 8.163595960941166,
 'tokens_per_second': 34.298610727388244,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.61031198501587,
  'reserved_gib': 19.970703125,
  'peak_allocated_gib': 19.02257490158081,
  'peak_reserved_gib': 19.970703125,
  'gpu_used_gib': 25.28076171875,
  'gpu_total_gib': 94.9600830078125}}

2. 下面尝试中文+思考模式。未翻车。

In [13]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从左到右的顺序排列，最左边的黄色的书作为第1本。现在请找到第14本书，并回答：
1.这本书的书脊是什么主要颜色？
2.如果书脊上有可辨认的文字，请准确抄写书名。
3.简要描述这本书的位置，说明你找到的是哪一行、哪一列。
请注意：
必须根据图片实际内容判断，不要根据书架的常见排列规律猜测。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="on",
)

Thinking mode: on
enable_thinking: True
Prompt verified: True

最终回答
1. 这本书的书脊主要颜色是白色。
2. 书脊上的书名是：苏东坡新传。
3. 这本书位于第一行，从左往右数第14本。<|im_end|>

性能统计
生成 Token: 505
Thinking Token: 458
Answer Token: 46
生成耗时: 14.237s
生成速度: 35.47 tokens/s
Peak VRAM: 19.02 GiB
Finish reason: eos


{'raw_generation': '1.  **确定目标位置**：\n    *   第一行书籍从左到右依次编号：\n        *   第1本：黄色书脊《苏东坡传》\n        *   第2本：浅蓝色书脊《数学Ⅱ(女孩)》\n        *   第3本：粉色书脊《数学Ⅱ(女孩)》\n        *   第4本：浅蓝色书脊《数学Ⅱ(女孩)》\n        *   第5本：粉色书脊《数学Ⅱ(女孩)》\n        *   第6本：红色书脊《现代汉语词典》\n        *   第7本：白色书脊《图解电的基础知识入门》\n        *   第8本：白色书脊《图解电机基础知识入门》\n        *   第9本：白色书脊《图解机械基础知识入门》\n        *   第10本：白色书脊《电磁场基本原理66课》\n        *   第11本：白色书脊《电工电路基本原理66课》\n        *   第12本：白色书脊《电子电路基本原理66课》\n        *   第13本：白色书脊《苏东坡新传》（上）\n        *   第14本：白色书脊《苏东坡新传》（下）\n    *   目标书籍是第14本，即《苏东坡新传》（下）。\n\n2.  **分析目标书籍特征**：\n    *   **书脊颜色**：书脊主体为白色，顶部和底部有棕色/米色区域。\n    *   **书名**：书脊上的文字清晰可见，为“苏东坡新传”。\n    *   **位置描述**：位于第一行，从左往右数第14本。\n\n3.  **构建回答**：\n    *   书脊主要颜色：白色。\n    *   书名：苏东坡新传。\n    *   位置：第一行，从左往右数第14本。\n</think>\n\n1. 这本书的书脊主要颜色是白色。\n2. 书脊上的书名是：苏东坡新传。\n3. 这本书位于第一行，从左往右数第14本。<|im_end|>',
 'generated_tokens': 505,
 'thinking_tokens': 458,
 'answer_tokens': 46,
 'closed_thinking': True,
 'finish_reason': 'eos',
 'truncated': False,
 

3. 尝试英文+非思考模式。没有翻车，但是有一定的列举分析。

In [14]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
Based on the image provided, here is the analysis of the bookshelf from left to right:

1.  **1st book:** Yellow spine, "Su Dongpo" (苏东坡).
2.  **2nd book:** Light blue spine, "Math II (Girls x Girls)" (数学Ⅱ(女×孩)).
3.  **3rd book:** Pink spine, "Math II (Girls x Girls) 3" (数学Ⅱ(女×孩)(3)).
4.  **4th book:** Light purple spine, "Math II (Girls x Girls) 2" (数学Ⅱ(女×孩)(2)).
5.  **5th book:** Light purple spine, "Math II (Girls x Girls) 4" (数学Ⅱ(女×孩)(4)).
6.  **6th book:** Red spine, "Modern Chinese Dictionary" (现代汉语词典).
7.  **7th book:** Green spine, "Illustrated Electric Basics Knowledge Introduction" (图解电的基础知识入门).
8.  **8th book:** Green spine, "Illustrated Electric Machine Basics Knowledge Introduction" (图解电机基础知识入门).
9.  **9th book:** Green spine, "Illustrated Mechanical Basics Knowledge Introduction" (图解机械基础知识入门).
10. **10th book:** White spine, "Electromagnetic Basic Principles 66 Lessons" (电磁场基本原理66课).
11. **11th book:** 

{'raw_generation': 'Based on the image provided, here is the analysis of the bookshelf from left to right:\n\n1.  **1st book:** Yellow spine, "Su Dongpo" (苏东坡).\n2.  **2nd book:** Light blue spine, "Math II (Girls x Girls)" (数学Ⅱ(女×孩)).\n3.  **3rd book:** Pink spine, "Math II (Girls x Girls) 3" (数学Ⅱ(女×孩)(3)).\n4.  **4th book:** Light purple spine, "Math II (Girls x Girls) 2" (数学Ⅱ(女×孩)(2)).\n5.  **5th book:** Light purple spine, "Math II (Girls x Girls) 4" (数学Ⅱ(女×孩)(4)).\n6.  **6th book:** Red spine, "Modern Chinese Dictionary" (现代汉语词典).\n7.  **7th book:** Green spine, "Illustrated Electric Basics Knowledge Introduction" (图解电的基础知识入门).\n8.  **8th book:** Green spine, "Illustrated Electric Machine Basics Knowledge Introduction" (图解电机基础知识入门).\n9.  **9th book:** Green spine, "Illustrated Mechanical Basics Knowledge Introduction" (图解机械基础知识入门).\n10. **10th book:** White spine, "Electromagnetic Basic Principles 66 Lessons" (电磁场基本原理66课).\n11. **11th book:** White spine, "Electrical Circuit Basic

4. 下面尝试英文+非思考模式+直接输入指令。翻大车了。

In [15]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试5.jpg")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from left to right, with the yellow book on the far left as the 1st book. Now please find the 14th book and answer:
What is the main color of this book's spine?
If there is legible text on the spine, please accurately copy the book title.
Briefly describe the position of this book, explaining which row and which column you found it in.
Please note:
You must judge based on the actual content of the image, and do not guess based on common arrangement patterns of bookshelves.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a book title.
Please output the answer directly, and do not output your analysis process.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
The 14th book is the one with the white spine and black text reading "电子电路基本原理66课". Its main color is white. It is located in the second row, fourth column.<|im_end|>

性能统计
生成 Token: 43
Thinking Token: 0
Answer Token: 43
生成耗时: 1.814s
生成速度: 23.71 tokens/s
Peak VRAM: 19.03 GiB
Finish reason: eos


{'raw_generation': 'The 14th book is the one with the white spine and black text reading "电子电路基本原理66课". Its main color is white. It is located in the second row, fourth column.<|im_end|>',
 'generated_tokens': 43,
 'thinking_tokens': 0,
 'answer_tokens': 43,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 1.8135161318350583,
 'tokens_per_second': 23.71084505131433,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 17.61031198501587,
  'reserved_gib': 19.970703125,
  'peak_allocated_gib': 19.02941131591797,
  'peak_reserved_gib': 19.970703125,
  'gpu_used_gib': 25.28076171875,
  'gpu_total_gib': 94.9600830078125}}

### 测试三 几何

1. 对于书架图的测试

In [6]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 2 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的上一行、同一列的书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，第 2 行、第 2 列的书是《Music》，位于书架的中间位置，形状为长方形，封面为白色，上面有乐谱和一把小提琴的图案。

位于这本书的上一行、同一列的书是《Ocean》，位于书架的上部，形状为长方形，封面为蓝色，上面有一只海龟和海洋的图案。<|im_end|>

性能统计
生成 Token: 81
Thinking Token: 0
Answer Token: 81
生成耗时: 3.720s
生成速度: 21.77 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，第 2 行、第 2 列的书是《Music》，位于书架的中间位置，形状为长方形，封面为白色，上面有乐谱和一把小提琴的图案。\n\n位于这本书的上一行、同一列的书是《Ocean》，位于书架的上部，形状为长方形，封面为蓝色，上面有一只海龟和海洋的图案。<|im_end|>',
 'generated_tokens': 81,
 'thinking_tokens': 0,
 'answer_tokens': 81,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 3.7200529570691288,
 'tokens_per_second': 21.77388358036076,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.65442943572998,
  'reserved_gib': 9.060546875,
  'peak_allocated_gib': 8.9859938621521,
  'peak_reserved_gib': 9.060546875,
  'gpu_used_gib': 24.12384033203125,
  'gpu_total_gib': 94.9600830078125}}

使用“左边的左边的”的描述，回答错误

In [7]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的同一行、左边的左边的书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，书架是一个 3 行 4 列的网格。

**第 2 行、第 3 列的书：**
*   **位置**：位于书架的中间层，从右数第二本。
*   **形状**：长方形，竖立放置。
*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。
*   **描述**：这本书的封面是一张雄狮的头部特写照片，背景为黑色。

**位于这本书同一行、左边的书（即第 2 行、第 2 列）：**
*   **位置**：位于书架的中间层，从左数第二本。
*   **形状**：长方形，竖立放置。
*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。
*   **描述**：这本书的封面是一张小提琴的图像，背景为白色，上面有乐谱。

**总结：**
1.  第 2 行第 3 列的书是一本封面为雄狮照片的长方形书籍，书名无法辨认。
2.  位于其左侧（第 2 行第 2 列）的书是一本封面为小提琴和乐谱的长方形书籍，书名无法辨认。<|im_end|>

性能统计
生成 Token: 278
Thinking Token: 0
Answer Token: 278
生成耗时: 8.388s
生成速度: 33.14 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，书架是一个 3 行 4 列的网格。\n\n**第 2 行、第 3 列的书：**\n*   **位置**：位于书架的中间层，从右数第二本。\n*   **形状**：长方形，竖立放置。\n*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。\n*   **描述**：这本书的封面是一张雄狮的头部特写照片，背景为黑色。\n\n**位于这本书同一行、左边的书（即第 2 行、第 2 列）：**\n*   **位置**：位于书架的中间层，从左数第二本。\n*   **形状**：长方形，竖立放置。\n*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。\n*   **描述**：这本书的封面是一张小提琴的图像，背景为白色，上面有乐谱。\n\n**总结：**\n1.  第 2 行第 3 列的书是一本封面为雄狮照片的长方形书籍，书名无法辨认。\n2.  位于其左侧（第 2 行第 2 列）的书是一本封面为小提琴和乐谱的长方形书籍，书名无法辨认。<|im_end|>',
 'generated_tokens': 278,
 'thinking_tokens': 0,
 'answer_tokens': 278,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 8.38837827090174,
 'tokens_per_second': 33.14109009179379,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.654431343078613,
  'reserved_gib': 9.060546875,
  'peak_allocated_gib': 8.98598051071167,
  'peak_reserved_gib': 9.060546875,
  'gpu_used_gib': 24.12384033203125,
  'gpu_total_gib': 94.9600830078125}}

“从它开始往左数的第三本书”回答正确。

In [9]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的同一行、从它开始往左数的第三本书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，书架是一个 3 行 4 列的网格。

**1. 第 2 行、第 3 列的书：**
*   **位置**：位于书架的中间层，从左数第三本。
*   **形状**：这是一本竖立的长方形书籍。
*   **书名**：封面上印有黑色的英文单词 "Animals"。
*   **封面内容**：封面背景为黑色，中央是一只雄狮的头部特写。

**2. 位于这本书的同一行（第 2 行），从它开始往左数的第三本书：**
*   **位置**：位于第 2 行，从左数第一本。
*   **形状**：这是一本竖立的长方形书籍。
*   **书名**：封面上印有白色的英文单词 "Space"。
*   **封面内容**：封面背景为深紫色星空，中央是一个明亮的螺旋星系图案。

---
**总结：**
*   第 2 行第 3 列的书是 **《Animals》**（动物）。
*   从它往左数第三本（即第 2 行第 1 列）的书是 **《Space》**（太空）。<|im_end|>

性能统计
生成 Token: 271
Thinking Token: 0
Answer Token: 271
生成耗时: 8.278s
生成速度: 32.74 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，书架是一个 3 行 4 列的网格。\n\n**1. 第 2 行、第 3 列的书：**\n*   **位置**：位于书架的中间层，从左数第三本。\n*   **形状**：这是一本竖立的长方形书籍。\n*   **书名**：封面上印有黑色的英文单词 "Animals"。\n*   **封面内容**：封面背景为黑色，中央是一只雄狮的头部特写。\n\n**2. 位于这本书的同一行（第 2 行），从它开始往左数的第三本书：**\n*   **位置**：位于第 2 行，从左数第一本。\n*   **形状**：这是一本竖立的长方形书籍。\n*   **书名**：封面上印有白色的英文单词 "Space"。\n*   **封面内容**：封面背景为深紫色星空，中央是一个明亮的螺旋星系图案。\n\n---\n**总结：**\n*   第 2 行第 3 列的书是 **《Animals》**（动物）。\n*   从它往左数第三本（即第 2 行第 1 列）的书是 **《Space》**（太空）。<|im_end|>',
 'generated_tokens': 271,
 'thinking_tokens': 0,
 'answer_tokens': 271,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 8.278496190905571,
 'tokens_per_second': 32.73541398710915,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.654431343078613,
  'reserved_gib': 9.060546875,
  'peak_allocated_gib': 8.986186504364014,
  'peak_reserved_gib': 9.060546875,
  'gpu_used_gib': 24.12384033203125,
  'gpu_total_gib': 94.9600830078125}}

“左边的（书a）左边的书”能回答。orc变弱。

In [10]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的同一行、左边的（书a）左边的书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，书架是一个 3 行 4 列的网格。

**第 2 行、第 3 列的书：**
*   **位置**：位于书架的中间层，从右数第二本。
*   **形状**：长方形，竖立放置。
*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。
*   **描述**：这本书的封面是一张雄狮的头部特写照片，背景为黑色。

**位于这本书同一行、左边的书（即第 2 行、第 2 列的书，书a）：**
*   **位置**：位于书架的中间层，从左数第二本。
*   **形状**：长方形，竖立放置。
*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。
*   **描述**：这本书的封面是一张小提琴的图像，背景为白色，上面有乐谱。

**位于书a左边的书（即第 2 行、第 1 列的书）：**
*   **位置**：位于书架的中间层，最左边。
*   **形状**：长方形，竖立放置。
*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。
*   **描述**：这本书的封面是一张螺旋星系的图像，背景为深紫色星空。

---

**总结回答：**

1.  **第 2 行、第 3 列的书**：位于书架中间层，从左数第三本。形状为长方形。封面上没有文字，只有一张雄狮的头部特写照片。
2.  **位于该书同一行、左边的书（书a）左边的书**：位于书架中间层，最左边。形状为长方形。封面上没有文字，只有一张螺旋星系的图像。<|im_end|>

性能统计
生成 Token: 410
Thinking Token: 0
Answer Token: 410
生成耗时: 12.362s
生成速度: 33.17 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，书架是一个 3 行 4 列的网格。\n\n**第 2 行、第 3 列的书：**\n*   **位置**：位于书架的中间层，从右数第二本。\n*   **形状**：长方形，竖立放置。\n*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。\n*   **描述**：这本书的封面是一张雄狮的头部特写照片，背景为黑色。\n\n**位于这本书同一行、左边的书（即第 2 行、第 2 列的书，书a）：**\n*   **位置**：位于书架的中间层，从左数第二本。\n*   **形状**：长方形，竖立放置。\n*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。\n*   **描述**：这本书的封面是一张小提琴的图像，背景为白色，上面有乐谱。\n\n**位于书a左边的书（即第 2 行、第 1 列的书）：**\n*   **位置**：位于书架的中间层，最左边。\n*   **形状**：长方形，竖立放置。\n*   **书名**：无法辨认（图片中该位置没有文字，只有图像）。\n*   **描述**：这本书的封面是一张螺旋星系的图像，背景为深紫色星空。\n\n---\n\n**总结回答：**\n\n1.  **第 2 行、第 3 列的书**：位于书架中间层，从左数第三本。形状为长方形。封面上没有文字，只有一张雄狮的头部特写照片。\n2.  **位于该书同一行、左边的书（书a）左边的书**：位于书架中间层，最左边。形状为长方形。封面上没有文字，只有一张螺旋星系的图像。<|im_end|>',
 'generated_tokens': 410,
 'thinking_tokens': 0,
 'answer_tokens': 410,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 12.362102560233325,
 'tokens_per_second': 33.16587918619093,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.65443229675293,
  'reserved_gib': 9.060

“上一行、左边的（书a）左边的书”能回答

In [11]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的上一行、左边的（书a）左边的书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个 3 行 4 列的二维网格。

**第 1 行：**
- 第 1 列：Travel
- 第 2 列：Ocean
- 第 3 列：Plants
- 第 4 列：Cooking

**第 2 行：**
- 第 1 列：Space
- 第 2 列：Music
- 第 3 列：Animals
- 第 4 列：History

**第 3 行：**
- 第 1 列：Art
- 第 2 列：Books
- 第 3 列：Health
- 第 4 列：Sports

---

**问题 1：找到第 2 行、第 3 列的那本书**

- **位置**：位于书架的中间层（第 2 行），从左边数第 3 个格子。
- **形状**：这是一本长方形的硬壳书，封面为黑色。
- **书名**：Animals（动物）。
- **封面内容**：封面上有一只雄狮的头部特写。

---

**问题 2：找到位于这本书的上一行、左边的（书a）左边的书**

- 首先，找到“第 2 行、第 3 列”的书（Animals）。
- 它的“上一行”是第 1 行。
- 它的“左边”是第 2 列。
- 所以，“上一行、左边的书”是第 1 行、第 2 列的书，即 **Ocean**（海洋）。
- 现在，找到 **Ocean** 这本书的“左边”的书。
- Ocean 的左边是第 1 列，即 **Travel**（旅行）。

- **位置**：位于书架的最顶层（第 1 行），最左边（第 1 列）。
- **形状**：这是一本长方形的硬壳书，封面为红色。
- **书名**：Travel（旅行）。
- **封面内容**：封面上有一座雪山和湖泊的风景图。

---

**最终答案：**

1. 第 2 行、第 3 列的书是 **Animals**（动物），位于书架中间层的第三格，是一本黑色封面的长方形书籍，封面上印有一只雄狮的头部。
2. 位于该书上一行、左边的书（即第 1 行、第 2 列的 Ocean）的左边，是 **Travel**（旅行），位于书架最顶层的最左侧，是一本红色封面的长方形书籍，封面上印有一座雪山和湖

{'raw_generation': '根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个 3 行 4 列的二维网格。\n\n**第 1 行：**\n- 第 1 列：Travel\n- 第 2 列：Ocean\n- 第 3 列：Plants\n- 第 4 列：Cooking\n\n**第 2 行：**\n- 第 1 列：Space\n- 第 2 列：Music\n- 第 3 列：Animals\n- 第 4 列：History\n\n**第 3 行：**\n- 第 1 列：Art\n- 第 2 列：Books\n- 第 3 列：Health\n- 第 4 列：Sports\n\n---\n\n**问题 1：找到第 2 行、第 3 列的那本书**\n\n- **位置**：位于书架的中间层（第 2 行），从左边数第 3 个格子。\n- **形状**：这是一本长方形的硬壳书，封面为黑色。\n- **书名**：Animals（动物）。\n- **封面内容**：封面上有一只雄狮的头部特写。\n\n---\n\n**问题 2：找到位于这本书的上一行、左边的（书a）左边的书**\n\n- 首先，找到“第 2 行、第 3 列”的书（Animals）。\n- 它的“上一行”是第 1 行。\n- 它的“左边”是第 2 列。\n- 所以，“上一行、左边的书”是第 1 行、第 2 列的书，即 **Ocean**（海洋）。\n- 现在，找到 **Ocean** 这本书的“左边”的书。\n- Ocean 的左边是第 1 列，即 **Travel**（旅行）。\n\n- **位置**：位于书架的最顶层（第 1 行），最左边（第 1 列）。\n- **形状**：这是一本长方形的硬壳书，封面为红色。\n- **书名**：Travel（旅行）。\n- **封面内容**：封面上有一座雪山和湖泊的风景图。\n\n---\n\n**最终答案：**\n\n1. 第 2 行、第 3 列的书是 **Animals**（动物），位于书架中间层的第三格，是一本黑色封面的长方形书籍，封面上印有一只雄狮的头部。\n2. 位于该书上一行、左边的书（即第 1 行、第 2 列的 Ocean）的左边，是 **Travel**（旅行），位于书架最顶层的最左侧，是一本红色封面的长方形书籍，封面上印有一座雪山和湖

“左边的（书a）左边的（书b）左边的书”不能回答

In [12]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 简要描述位于这本书的上一行、左边的（书a）左边的（书b）左边的书位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，按照从上到下、从左到右的顺序排列，第 2 行、第 3 列的书是《Animals》。

1.  **第 2 行、第 3 列的书**：
    *   **位置**：位于书架的中间层，从左数第三本。
    *   **形状**：长方形，竖立放置。
    *   **书名**：Animals。

2.  **位于这本书的上一行、左边的（书a）左边的（书b）左边的书**：
    *   首先找到第 2 行、第 3 列的书（《Animals》）。
    *   它的上一行是第 1 行，左边的书是第 1 行、第 2 列的书（《Ocean》）。
    *   再往左，即第 1 行、第 1 列的书（《Travel》）。
    *   **位置**：位于书架的最顶层，最左侧。
    *   **形状**：长方形，竖立放置。
    *   **书名**：Travel。<|im_end|>

性能统计
生成 Token: 252
Thinking Token: 0
Answer Token: 252
生成耗时: 7.717s
生成速度: 32.66 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，按照从上到下、从左到右的顺序排列，第 2 行、第 3 列的书是《Animals》。\n\n1.  **第 2 行、第 3 列的书**：\n    *   **位置**：位于书架的中间层，从左数第三本。\n    *   **形状**：长方形，竖立放置。\n    *   **书名**：Animals。\n\n2.  **位于这本书的上一行、左边的（书a）左边的（书b）左边的书**：\n    *   首先找到第 2 行、第 3 列的书（《Animals》）。\n    *   它的上一行是第 1 行，左边的书是第 1 行、第 2 列的书（《Ocean》）。\n    *   再往左，即第 1 行、第 1 列的书（《Travel》）。\n    *   **位置**：位于书架的最顶层，最左侧。\n    *   **形状**：长方形，竖立放置。\n    *   **书名**：Travel。<|im_end|>',
 'generated_tokens': 252,
 'thinking_tokens': 0,
 'answer_tokens': 252,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 7.71682555694133,
 'tokens_per_second': 32.65591507032636,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.654430866241455,
  'reserved_gib': 9.060546875,
  'peak_allocated_gib': 8.98648738861084,
  'peak_reserved_gib': 9.060546875,
  'gpu_used_gib': 24.12384033203125,
  'gpu_total_gib': 94.9600830078125}}

In [16]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 现在假设一本书c,位于这本书(即第 2 行、第 3 列的那本书)的上一行、左边的左边的左边的位置。简要描述书c的位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
要找的书不一定存在，请仔细辨认，诚实回答。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个3行4列的二维网格：

- 第1行：Travel, Ocean, Plants, Cooking
- 第2行：Space, Music, Animals, History
- 第3行：Art, Books, Health, Sports

---

**第2行、第3列的书：**

- 位置：第2行第3列
- 形状：矩形（标准书籍形状）
- 书名：Animals
- 封面内容：一只雄狮的头部特写

---

**书c的位置：**

题目要求：位于“这本书（第2行、第3列）的上一行、左边的左边的左边的位置”。

- 上一行 = 第1行
- 左边的左边 = 向左移动两列
- 第2行第3列 → 上一行第3列 = 第1行第3列（Plants）
- 再向左两列：第1行第3列 - 2 = 第1行第1列

所以书c位于第1行第1列。

- 位置：第1行第1列
- 形状：矩形（标准书籍形状）
- 书名：Travel
- 封面内容：雪山湖泊风景图

---

**最终答案：**

1. 第2行第3列的书是一本矩形书籍，书名为“Animals”，封面是一只雄狮的头部特写。

2. 书c位于第1行第1列，是一本矩形书籍，书名为“Travel”，封面是雪山湖泊的风景图。<|im_end|>

性能统计
生成 Token: 354
Thinking Token: 0
Answer Token: 354
生成耗时: 10.853s
生成速度: 32.62 tokens/s
Peak VRAM: 8.99 GiB
Finish reason: eos


{'raw_generation': '根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个3行4列的二维网格：\n\n- 第1行：Travel, Ocean, Plants, Cooking\n- 第2行：Space, Music, Animals, History\n- 第3行：Art, Books, Health, Sports\n\n---\n\n**第2行、第3列的书：**\n\n- 位置：第2行第3列\n- 形状：矩形（标准书籍形状）\n- 书名：Animals\n- 封面内容：一只雄狮的头部特写\n\n---\n\n**书c的位置：**\n\n题目要求：位于“这本书（第2行、第3列）的上一行、左边的左边的左边的位置”。\n\n- 上一行 = 第1行\n- 左边的左边 = 向左移动两列\n- 第2行第3列 → 上一行第3列 = 第1行第3列（Plants）\n- 再向左两列：第1行第3列 - 2 = 第1行第1列\n\n所以书c位于第1行第1列。\n\n- 位置：第1行第1列\n- 形状：矩形（标准书籍形状）\n- 书名：Travel\n- 封面内容：雪山湖泊风景图\n\n---\n\n**最终答案：**\n\n1. 第2行第3列的书是一本矩形书籍，书名为“Animals”，封面是一只雄狮的头部特写。\n\n2. 书c位于第1行第1列，是一本矩形书籍，书名为“Travel”，封面是雪山湖泊的风景图。<|im_end|>',
 'generated_tokens': 354,
 'thinking_tokens': 0,
 'answer_tokens': 354,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 10.853229978121817,
 'tokens_per_second': 32.61701822532104,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.654433250427246,
  'reserved_gib': 9.060546875,
  'peak_allocated_gib': 8.9911322593688

上一格模型不能正确识别；这里模型能理解，但是陷入怀疑，循环。

In [17]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
请仔细观察图片中的书架。
我们将书架中的书籍按照从上到下、从左到右的顺序排列，
定义为一个二维网格：
行：从上往下编号，第 1 行、第 2 行……
列：从左往右编号，第 1 列、第 2 列……
现在请找到第 2 行、第 3 列的那本书，并回答：
1. 简要描述这本书的位置、形状、书名（如果有）。
2. 现在假设一本书c,位于这本书(即第 2 行、第 3 列的那本书，设为书a)的上一行、从书a开始往左数的第三个位置。简要描述书c的位置、形状、书名（如果有）。
请注意：
必须根据图片实际内容判断，不要猜测。
不要将行和列的顺序颠倒。
如果文字无法辨认，请明确回答“无法辨认”，不要编造书名。
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个3行4列的二维网格。

**第1行：**
- 第1列：Travel
- 第2列：Ocean
- 第3列：Plants
- 第4列：Cooking

**第2行：**
- 第1列：Space
- 第2列：Music
- 第3列：Animals
- 第4列：History

**第3行：**
- 第1列：Art
- 第2列：Books
- 第3列：Health
- 第4列：Sports

---

**问题1：找到第2行、第3列的那本书**

- **位置**：第2行、第3列。
- **形状**：矩形，竖立放置。
- **书名**：Animals。
- **描述**：这本书的封面是黑色的，上面有一只雄狮的图像，书名“Animals”位于封面顶部。

---

**问题2：找到书c**

- **位置**：书a（第2行、第3列）的上一行、从书a开始往左数的第三个位置。
  - 书a的上一行是第1行。
  - 从书a（第3列）往左数：
    - 第1个位置：第2列（Ocean）
    - 第2个位置：第1列（Travel）
    - 第3个位置：第0列（不存在）

  等等，这里需要仔细计算。

  从书a（第3列）往左数：
  - 第1个：第2列
  - 第2个：第1列
  - 第3个：第0列（不存在）

  所以，从书a开始往左数的第三个位置，实际上超出了书架的左边界。

  但是，题目要求“从书a开始往左数的第三个位置”，这意味着：
  - 第1个：第2列
  - 第2个：第1列
  - 第3个：第0列（不存在）

  因此，书c的位置不存在于书架中。

  但是，题目要求“简要描述书c的位置、形状、书名（如果有）”，如果书c不存在，应该如何回答？

  根据题目要求：“如果文字无法辨认，请明确回答‘无法辨认’，不要编造书名。”

  但是，书c不存在，所以无法描述。

  等等，让我重新计算。

  从书a（第3列）往左数：
  - 第1个：第2列
  - 第2个：第1列
  - 第3个：第0列（不存在）

  所以，书c不存在。

  但是，题目要求

{'raw_generation': '根据图片内容，我们按照从上到下、从左到右的顺序排列书架中的书籍，构建一个3行4列的二维网格。\n\n**第1行：**\n- 第1列：Travel\n- 第2列：Ocean\n- 第3列：Plants\n- 第4列：Cooking\n\n**第2行：**\n- 第1列：Space\n- 第2列：Music\n- 第3列：Animals\n- 第4列：History\n\n**第3行：**\n- 第1列：Art\n- 第2列：Books\n- 第3列：Health\n- 第4列：Sports\n\n---\n\n**问题1：找到第2行、第3列的那本书**\n\n- **位置**：第2行、第3列。\n- **形状**：矩形，竖立放置。\n- **书名**：Animals。\n- **描述**：这本书的封面是黑色的，上面有一只雄狮的图像，书名“Animals”位于封面顶部。\n\n---\n\n**问题2：找到书c**\n\n- **位置**：书a（第2行、第3列）的上一行、从书a开始往左数的第三个位置。\n  - 书a的上一行是第1行。\n  - 从书a（第3列）往左数：\n    - 第1个位置：第2列（Ocean）\n    - 第2个位置：第1列（Travel）\n    - 第3个位置：第0列（不存在）\n\n  等等，这里需要仔细计算。\n\n  从书a（第3列）往左数：\n  - 第1个：第2列\n  - 第2个：第1列\n  - 第3个：第0列（不存在）\n\n  所以，从书a开始往左数的第三个位置，实际上超出了书架的左边界。\n\n  但是，题目要求“从书a开始往左数的第三个位置”，这意味着：\n  - 第1个：第2列\n  - 第2个：第1列\n  - 第3个：第0列（不存在）\n\n  因此，书c的位置不存在于书架中。\n\n  但是，题目要求“简要描述书c的位置、形状、书名（如果有）”，如果书c不存在，应该如何回答？\n\n  根据题目要求：“如果文字无法辨认，请明确回答‘无法辨认’，不要编造书名。”\n\n  但是，书c不存在，所以无法描述。\n\n  等等，让我重新计算。\n\n  从书a（第3列）往左数：\n  - 第1个：第2列\n  - 第2个：第1列\n  - 第3个：第0列（不存在）\n\n  所

换成英文，反而答错了。

In [18]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from top to bottom and from left to right,
and define this as a two-dimensional grid:
Rows: numbered from top to bottom, row 1, row 2, ...
Columns: numbered from left to right, column 1, column 2, ...
Now please find the book in row 2, column 3, and answer:
1. Briefly describe the position, shape, and title (if any) of this book.
2. Now suppose there is a book c, located one row above this book (i.e., the book in row 2, column 3, designated as book a), at the third position counting leftward starting from book a. Briefly describe the position, shape, and title (if any) of book c.
Please note:
You must judge based on the actual content of the image; do not guess.
Do not reverse the order of rows and columns.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a title.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
Based on the image provided, here is the analysis:

1.  **Book in row 2, column 3:**
    *   **Position:** It is located in the middle row, third from the left.
    *   **Shape:** It is a standard rectangular hardcover book.
    *   **Title:** The title on the cover is "Animals".

2.  **Book c (one row above book a, at the third position counting leftward starting from book a):**
    *   **Position:** "One row above" means we look at row 1. "Counting leftward starting from book a" means we start at column 3 and move left to column 2, then column 1. So, we are looking for the book in row 1, column 2.
    *   **Shape:** It is a standard rectangular hardcover book.
    *   **Title:** The title on the cover is "Ocean".<|im_end|>

性能统计
生成 Token: 205
Thinking Token: 0
Answer Token: 205
生成耗时: 6.291s
生成速度: 32.59 tokens/s
Peak VRAM: 9.00 GiB
Finish reason: eos


{'raw_generation': 'Based on the image provided, here is the analysis:\n\n1.  **Book in row 2, column 3:**\n    *   **Position:** It is located in the middle row, third from the left.\n    *   **Shape:** It is a standard rectangular hardcover book.\n    *   **Title:** The title on the cover is "Animals".\n\n2.  **Book c (one row above book a, at the third position counting leftward starting from book a):**\n    *   **Position:** "One row above" means we look at row 1. "Counting leftward starting from book a" means we start at column 3 and move left to column 2, then column 1. So, we are looking for the book in row 1, column 2.\n    *   **Shape:** It is a standard rectangular hardcover book.\n    *   **Title:** The title on the cover is "Ocean".<|im_end|>',
 'generated_tokens': 205,
 'thinking_tokens': 0,
 'answer_tokens': 205,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 6.291063018608838,
 'tokens_per_second': 32.58590788132532,
 'mask_

加入“要找的书不一定存在，请仔细辨认，诚实回答。”，模型变得自信。

In [19]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
Please carefully observe the bookshelf in the image.
We arrange the books in the bookshelf in order from top to bottom and from left to right,
and define this as a two-dimensional grid:
Rows: numbered from top to bottom, row 1, row 2, ...
Columns: numbered from left to right, column 1, column 2, ...
Now please find the book in row 2, column 3, and answer:
1. Briefly describe the position, shape, and title (if any) of this book.
2. Now suppose there is a book c, located one row above this book (i.e., the book in row 2, column 3, designated as book a), at the third position counting leftward starting from book a. Briefly describe the position, shape, and title (if any) of book c.
Please note:
You must judge based on the actual content of the image; do not guess.
Do not reverse the order of rows and columns.
If the text cannot be recognized, clearly answer "cannot be recognized" and do not fabricate a title.
The book you are looking for may not exist. Please identify it carefully and answer honestly.
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
Based on the image provided, here is the analysis:

1.  **Book in row 2, column 3:**
    *   **Position:** It is located in the middle row, third from the left.
    *   **Shape:** It is a standard rectangular hardcover book.
    *   **Title:** The title on the cover is "Animals". The cover features a close-up image of a lion's face.

2.  **Book c (one row above book a, at the third position counting leftward starting from book a):**
    *   **Target Location:** The book in row 2, column 3 is "Animals". One row above it is row 1. Counting three positions to the left from the "Animals" book (which is in column 3) leads to column 1. Therefore, the target book is in **row 1, column 1**.
    *   **Position:** It is located in the top row, first from the left.
    *   **Shape:** It is a standard rectangular hardcover book.
    *   **Title:** The title on the cover is "Travel". The cover features an image of a mountain land

{'raw_generation': 'Based on the image provided, here is the analysis:\n\n1.  **Book in row 2, column 3:**\n    *   **Position:** It is located in the middle row, third from the left.\n    *   **Shape:** It is a standard rectangular hardcover book.\n    *   **Title:** The title on the cover is "Animals". The cover features a close-up image of a lion\'s face.\n\n2.  **Book c (one row above book a, at the third position counting leftward starting from book a):**\n    *   **Target Location:** The book in row 2, column 3 is "Animals". One row above it is row 1. Counting three positions to the left from the "Animals" book (which is in column 3) leads to column 1. Therefore, the target book is in **row 1, column 1**.\n    *   **Position:** It is located in the top row, first from the left.\n    *   **Shape:** It is a standard rectangular hardcover book.\n    *   **Title:** The title on the cover is "Travel". The cover features an image of a mountain landscape with a lake.\n\n**Summary:**\n* 

In [20]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
假设你是一个无人机，你按顺序执行了以下任务：
1.往前飞了5米；
2.继续往前飞了5米；
3.然后左转，飞了5米；
4.最后右转，飞了5米。
请回答下述问题：
最终你的位置相对于最开始的位置在{答案1}{答案2}。
答案1候选项：A.左; B.右
答案2候选项：A.前; B.后
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
我们来逐步分析无人机的运动轨迹：

1. **往前飞了5米** → 当前位置：起点 + 前5米  
2. **继续往前飞了5米** → 当前位置：起点 + 前10米（仍在正前方）  
3. **然后左转，飞了5米** → 左转90度，向前飞5米 → 此时位置在起点的**左侧**5米处（相对于原前进方向）  
4. **最后右转，飞了5米** → 右转90度（即回到原来的朝向），再向前飞5米 → 此时位置在起点的**右侧**5米处（因为左转后向左飞，再右转就向右飞了）

---

让我们用坐标系更清晰地建模：

设起点为原点 (0, 0)，初始朝向为 +y 方向（前）。

- 步骤1：前5米 → (0, 5)
- 步骤2：前5米 → (0, 10)
- 步骤3：左转90度 → 新朝向为 -x 方向（左），飞5米 → (0 - 5, 10) = (-5, 10)
- 步骤4：右转90度 → 新朝向为 +y 方向（前），飞5米 → (-5, 10 + 5) = (-5, 15)

最终位置：(-5, 15)

相对于起点 (0, 0)：

- x 方向：-5 → 在**左**侧
- y 方向：+15 → 在**前**方

所以最终位置相对于最开始的位置：**左** 和 **前**

---

答案1：A.左  
答案2：A.前

---

✅ 最终答案：

{A}{A}<|im_end|>

性能统计
生成 Token: 401
Thinking Token: 0
Answer Token: 401
生成耗时: 11.748s
生成速度: 34.13 tokens/s
Peak VRAM: 8.98 GiB
Finish reason: eos


{'raw_generation': '我们来逐步分析无人机的运动轨迹：\n\n1. **往前飞了5米** → 当前位置：起点 + 前5米  \n2. **继续往前飞了5米** → 当前位置：起点 + 前10米（仍在正前方）  \n3. **然后左转，飞了5米** → 左转90度，向前飞5米 → 此时位置在起点的**左侧**5米处（相对于原前进方向）  \n4. **最后右转，飞了5米** → 右转90度（即回到原来的朝向），再向前飞5米 → 此时位置在起点的**右侧**5米处（因为左转后向左飞，再右转就向右飞了）\n\n---\n\n让我们用坐标系更清晰地建模：\n\n设起点为原点 (0, 0)，初始朝向为 +y 方向（前）。\n\n- 步骤1：前5米 → (0, 5)\n- 步骤2：前5米 → (0, 10)\n- 步骤3：左转90度 → 新朝向为 -x 方向（左），飞5米 → (0 - 5, 10) = (-5, 10)\n- 步骤4：右转90度 → 新朝向为 +y 方向（前），飞5米 → (-5, 10 + 5) = (-5, 15)\n\n最终位置：(-5, 15)\n\n相对于起点 (0, 0)：\n\n- x 方向：-5 → 在**左**侧\n- y 方向：+15 → 在**前**方\n\n所以最终位置相对于最开始的位置：**左** 和 **前**\n\n---\n\n答案1：A.左  \n答案2：A.前\n\n---\n\n✅ 最终答案：\n\n{A}{A}<|im_end|>',
 'generated_tokens': 401,
 'thinking_tokens': 0,
 'answer_tokens': 401,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 11.748438237700611,
 'tokens_per_second': 34.132196287434645,
 'mask_tokens': 0,
 'vram': {'allocated_gib': 8.654429912567139,
  'reserved_gib': 9.060546875,
  'peak_allocate

In [21]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
假设你是一个无人机，你按顺序执行了以下任务：
1.往前飞了5米；
2.然后向上飞了5米；
3.继续往前飞了5米；
4.然后左转，飞了5米；
5.然后向下飞了5米；
6.最后右转，飞了5米。
请回答下述问题：
最终你的位置相对于最开始的位置在{答案1}{答案2}{答案3}方位。
答案1候选项：A.左; B.右
答案2候选项：A.前; B.后
答案2候选项：A.上; B.下；C.同一水平
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
我们来逐步分析无人机的飞行路径，建立坐标系：

设初始位置为原点 (0, 0, 0)，其中：
- x轴：前后方向（前为正）
- y轴：左右方向（右为正）
- z轴：上下方向（上为正）

**步骤1：往前飞了5米**
→ 位置：(5, 0, 0)

**步骤2：向上飞了5米**
→ 位置：(5, 0, 5)

**步骤3：继续往前飞了5米**
→ 位置：(10, 0, 5)

**步骤4：左转，飞了5米**
注意：“左转”是相对于当前朝向。当前朝向是“前”（即x轴正方向）。左转90度后，朝向变为“左”（即y轴负方向）。
→ 飞5米向左（y减5）
→ 位置：(10, -5, 5)

**步骤5：向下飞了5米**
→ 位置：(10, -5, 0)

**步骤6：右转，飞了5米**
当前朝向是“左”（y轴负方向）。右转90度后，朝向变为“后”（x轴负方向）。
→ 飞5米向后（x减5）
→ 位置：(5, -5, 0)

---

最终位置：(5, -5, 0)

初始位置：(0, 0, 0)

比较：
- x方向：+5 → 前
- y方向：-5 → 左
- z方向：0 → 同一水平

所以相对于初始位置：
- 前后：前（x+5）
- 左右：左（y-5）
- 上下：同一水平

题目问的是“方位”，选项是：
答案1：A.左; B.右 → 对应左右方向 → 左
答案2：A.前; B.后 → 对应前后方向 → 前
答案3：A.上; B.下; C.同一水平 → 对应上下方向 → 同一水平

---

因此最终答案：

**答案1：A.左**  
**答案2：A.前**  
**答案3：C.同一水平**

---

✅ 最终输出：

A.左; A.前; C.同一水平<|im_end|>

性能统计
生成 Token: 538
Thinking Token: 0
Answer Token: 538
生成耗时: 15.646s
生成速度: 34.39 tokens/s
Peak VRAM: 8.98 GiB
Finish reason: eos


{'raw_generation': '我们来逐步分析无人机的飞行路径，建立坐标系：\n\n设初始位置为原点 (0, 0, 0)，其中：\n- x轴：前后方向（前为正）\n- y轴：左右方向（右为正）\n- z轴：上下方向（上为正）\n\n**步骤1：往前飞了5米**\n→ 位置：(5, 0, 0)\n\n**步骤2：向上飞了5米**\n→ 位置：(5, 0, 5)\n\n**步骤3：继续往前飞了5米**\n→ 位置：(10, 0, 5)\n\n**步骤4：左转，飞了5米**\n注意：“左转”是相对于当前朝向。当前朝向是“前”（即x轴正方向）。左转90度后，朝向变为“左”（即y轴负方向）。\n→ 飞5米向左（y减5）\n→ 位置：(10, -5, 5)\n\n**步骤5：向下飞了5米**\n→ 位置：(10, -5, 0)\n\n**步骤6：右转，飞了5米**\n当前朝向是“左”（y轴负方向）。右转90度后，朝向变为“后”（x轴负方向）。\n→ 飞5米向后（x减5）\n→ 位置：(5, -5, 0)\n\n---\n\n最终位置：(5, -5, 0)\n\n初始位置：(0, 0, 0)\n\n比较：\n- x方向：+5 → 前\n- y方向：-5 → 左\n- z方向：0 → 同一水平\n\n所以相对于初始位置：\n- 前后：前（x+5）\n- 左右：左（y-5）\n- 上下：同一水平\n\n题目问的是“方位”，选项是：\n答案1：A.左; B.右 → 对应左右方向 → 左\n答案2：A.前; B.后 → 对应前后方向 → 前\n答案3：A.上; B.下; C.同一水平 → 对应上下方向 → 同一水平\n\n---\n\n因此最终答案：\n\n**答案1：A.左**  \n**答案2：A.前**  \n**答案3：C.同一水平**\n\n---\n\n✅ 最终输出：\n\nA.左; A.前; C.同一水平<|im_end|>',
 'generated_tokens': 538,
 'thinking_tokens': 0,
 'answer_tokens': 538,
 'closed_thinking': False,
 'finish_reason': 'eos',
 'truncated': False,
 'generate_seconds': 15.64

In [22]:
IMAGE_PATH = (PROJECT_ROOT/ "datasets"/ "几何简单测试"/ "几何测试1.png")
image = Image.open(IMAGE_PATH).convert("RGB")
# display(image)

question = """
假设你是一个无人机，你按顺序执行了以下任务：
1.往前飞了3米；
2.然后向上飞了6米；
3.继续往前飞了6米；
4.然后左转，飞了3米；
5.然后向下飞了9米；
6.最后右转，飞了3米。
请回答下述问题：
最终你的位置相对于最开始的位置在{答案1}{答案2}{答案3}方位。
答案1候选项：A.左; B.右
答案2候选项：A.前; B.后
答案2候选项：A.上; B.下；C.同一水平
"""

test_xiaomi(
    image=image,
    question=question,
    mode="off",
)

Thinking mode: off
enable_thinking: False
Prompt verified: True

最终回答
我们来逐步分析无人机的飞行路径，建立坐标系来追踪其最终位置。

假设无人机初始位置为原点 (0, 0)，其中：
- “前”方向为 +y 轴（向上在图像中，但这里“前”是飞行方向，我们定义一个局部坐标系）
- “右”为 +x 轴
- “上”为 +z 轴（垂直方向）

但注意：题目中的“前”、“后”、“上”、“下”、“左”、“右”是相对于无人机当前朝向的。我们需要仔细跟踪朝向变化。

---

**初始状态：**
位置：(0, 0, 0)
朝向：前 = +y，右 = +x，上 = +z

---

**任务1：往前飞了3米**
→ 位置：(0, 3, 0)
朝向不变：前 = +y

---

**任务2：然后向上飞了6米**
→ 位置：(0, 3, 6)
朝向不变：前 = +y

---

**任务3：继续往前飞了6米**
→ 位置：(0, 9, 6)
朝向不变：前 = +y

---

**任务4：然后左转，飞了3米**
左转90度 → 新朝向：前 = -x（因为原来前是+y，左转90度后，前指向-x方向）
飞3米 → 位置：(-3, 9, 6)

---

**任务5：然后向下飞了9米**
→ 位置：(-3, 9, -3)
朝向不变：前 = -x

---

**任务6：最后右转，飞了3米**
右转90度 → 新朝向：前 = +y（因为原来前是-x，右转90度后，前指向+y方向）
飞3米 → 位置：(-3, 12, -3)

---

**最终位置：(-3, 12, -3)**

相对于初始位置 (0, 0, 0)：

- x方向：-3 → 向左
- y方向：+12 → 向前
- z方向：-3 → 向下

所以最终位置在：
- 左（x负）
- 前（y正）
- 下（z负）

---

现在匹配选项：

答案1候选项：A.左; B.右 → 选 **A.左**

答案2候选项：A.前; B.后 → 选 **A.前**

答案2候选项：A.上; B.下；C.同一水平 → 选 **B.下**

---

✅ 最终答案：

**A.左, A.前, B.下**

---

**答案：**

{A} {A} {B}<|im_end|>

性能统

{'raw_generation': '我们来逐步分析无人机的飞行路径，建立坐标系来追踪其最终位置。\n\n假设无人机初始位置为原点 (0, 0)，其中：\n- “前”方向为 +y 轴（向上在图像中，但这里“前”是飞行方向，我们定义一个局部坐标系）\n- “右”为 +x 轴\n- “上”为 +z 轴（垂直方向）\n\n但注意：题目中的“前”、“后”、“上”、“下”、“左”、“右”是相对于无人机当前朝向的。我们需要仔细跟踪朝向变化。\n\n---\n\n**初始状态：**\n位置：(0, 0, 0)\n朝向：前 = +y，右 = +x，上 = +z\n\n---\n\n**任务1：往前飞了3米**\n→ 位置：(0, 3, 0)\n朝向不变：前 = +y\n\n---\n\n**任务2：然后向上飞了6米**\n→ 位置：(0, 3, 6)\n朝向不变：前 = +y\n\n---\n\n**任务3：继续往前飞了6米**\n→ 位置：(0, 9, 6)\n朝向不变：前 = +y\n\n---\n\n**任务4：然后左转，飞了3米**\n左转90度 → 新朝向：前 = -x（因为原来前是+y，左转90度后，前指向-x方向）\n飞3米 → 位置：(-3, 9, 6)\n\n---\n\n**任务5：然后向下飞了9米**\n→ 位置：(-3, 9, -3)\n朝向不变：前 = -x\n\n---\n\n**任务6：最后右转，飞了3米**\n右转90度 → 新朝向：前 = +y（因为原来前是-x，右转90度后，前指向+y方向）\n飞3米 → 位置：(-3, 12, -3)\n\n---\n\n**最终位置：(-3, 12, -3)**\n\n相对于初始位置 (0, 0, 0)：\n\n- x方向：-3 → 向左\n- y方向：+12 → 向前\n- z方向：-3 → 向下\n\n所以最终位置在：\n- 左（x负）\n- 前（y正）\n- 下（z负）\n\n---\n\n现在匹配选项：\n\n答案1候选项：A.左; B.右 → 选 **A.左**\n\n答案2候选项：A.前; B.后 → 选 **A.前**\n\n答案2候选项：A.上; B.下；C.同一水平 → 选 **B.下**\n\n---\n\n✅ 最终答案：\n\n**A.左, A.前, B.下**\n\n---\n\n*